In [2]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe

from utils.melt_stage_writer import (
    build_sensor_messages_stage,
    validate_sensor_messages_stage,
)



In [3]:
import os
import gc
import psutil

def memory_gb() -> float:
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 3)

def log_memory(label: str) -> None:
    print(f"[memory] {label}: {memory_gb():.2f} GB")

In [4]:

engine = get_engine_from_env()


In [5]:
log_memory("before read")
# read chunk
log_memory("after read")
# melt chunk
log_memory("after melt")
# write chunk
log_memory("after write")
gc.collect()
log_memory("after gc")

[memory] before read: 0.17 GB
[memory] after read: 0.17 GB
[memory] after melt: 0.17 GB
[memory] after write: 0.17 GB
[memory] after gc: 0.17 GB


In [6]:

melt_table_name = build_sensor_messages_stage(
    engine=engine,
    schema="capstone",
    source_table="synthetic_observations_premelt_stage",
    target_table="synthetic_sensor_messages_stage",
    if_exists="replace",
    random_seed=42,
    n_sensors=52,
    chunk_size=10000,
)


[chunk] 1 | source rows 0 to 9,999
[chunk] 1 melted 10,000 observations -> 520,000 sensor rows
[chunk] 1 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 2 | source rows 10,000 to 19,999
[chunk] 2 melted 10,000 observations -> 520,000 sensor rows
[chunk] 2 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 3 | source rows 20,000 to 29,999
[chunk] 3 melted 10,000 observations -> 520,000 sensor rows
[chunk] 3 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 4 | source rows 30,000 to 39,999
[chunk] 4 melted 10,000 observations -> 520,000 sensor rows
[chunk] 4 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 5 | source rows 40,000 to 49,999
[chunk] 5 melted 10,000 observations -> 520,000 sensor rows
[chunk] 5 wrote 520,000 rows to capstone.synthetic_sensor_messages_stage
[chunk] 6 | source rows 50,000 to 59,999
[chunk] 6 melted 10,000 observations -> 520,000 sensor rows
[chunk] 6 wrote 520,000 rows to cap

In [7]:
log_memory("before read")
# read chunk
log_memory("after read")
# melt chunk
log_memory("after melt")
# write chunk
log_memory("after write")
gc.collect()
log_memory("after gc")

[memory] before read: 0.67 GB
[memory] after read: 0.67 GB
[memory] after melt: 0.67 GB
[memory] after write: 0.67 GB
[memory] after gc: 0.67 GB


In [8]:

print("Built table:", melt_table_name)


Built table: synthetic_sensor_messages_stage


In [10]:

validation_dataframe = validate_sensor_messages_stage(
    engine=engine,
    schema="capstone",
    table_name="synthetic_sensor_messages_stage",
)


In [11]:
display(validation_dataframe)

,row_count,distinct_observation_count,distinct_sensor_name_count,min_sensor_index,max_sensor_index,min_message_sequence_index,max_message_sequence_index
0,39000000,750000,52,0,51,0,51


----

In [12]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        generated_row_id,
        sensor_name,
        sensor_index,
        sensor_value,
        message_sequence_index,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_stage
    ORDER BY observation_index, message_sequence_index
    LIMIT 104
    """
)


In [13]:

display(sample_dataframe)

,observation_index,generated_row_id,sensor_name,sensor_index,sensor_value,message_sequence_index,stream_state,phase,meta_episode_id
0,1,premelt_run_001_obs_000000000001,sensor_41,41,36.767719,0,normal,normal,0
1,1,premelt_run_001_obs_000000000001,sensor_49,49,53.702847,1,normal,normal,0
2,1,premelt_run_001_obs_000000000001,sensor_46,46,43.532345,2,normal,normal,0
3,1,premelt_run_001_obs_000000000001,sensor_32,32,928.790527,3,normal,normal,0
4,1,premelt_run_001_obs_000000000001,sensor_18,18,2.743295,4,normal,normal,0
...,...,...,...,...,...,...,...,...,...
99,2,premelt_run_001_obs_000000000002,sensor_13,13,4.297912,47,normal,normal,0
100,2,premelt_run_001_obs_000000000002,sensor_11,11,43.319096,48,normal,normal,0
101,2,premelt_run_001_obs_000000000002,sensor_07,7,15.981363,49,normal,normal,0
102,2,premelt_run_001_obs_000000000002,sensor_08,8,15.264808,50,normal,normal,0


In [14]:
sequence_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_count,
        MIN(message_sequence_index) AS min_msg_seq,
        MAX(message_sequence_index) AS max_msg_seq,
        COUNT(DISTINCT message_sequence_index) AS distinct_msg_seq_count
    FROM capstone.synthetic_sensor_messages_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sequence_check_dataframe)

,observation_index,message_count,distinct_sensor_count,min_msg_seq,max_msg_seq,distinct_msg_seq_count
0,1,52,52,0,51,52
1,2,52,52,0,51,52
2,3,52,52,0,51,52
3,4,52,52,0,51,52
4,5,52,52,0,51,52
5,6,52,52,0,51,52
6,7,52,52,0,51,52
7,8,52,52,0,51,52
8,9,52,52,0,51,52
9,10,52,52,0,51,52
